# ISIC 2019 exploratory audit, split, and baseline

This is an exported record of the Kaggle work, including historical attempts and session resets. The direct-URL ground-truth cell failed in Kaggle; later cells use the uploaded CSV. Run the relevant cells with the three Kaggle datasets attached, or refactor this record before using **Run All**. Outputs from the long input-file listing and obsolete error traceback were cleared; useful split and training outputs are retained. The held-out test set has not been evaluated.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [9]:
from pathlib import Path
from collections import Counter

root = Path("/kaggle/input")
images = [
    p for p in root.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("Dataset folders:", [p.name for p in root.iterdir()])
print("Total images:", len(images))
print("\nImages by folder:")
for folder, count in Counter(p.parent.name for p in images).most_common():
    print(f"{folder}: {count}")

Dataset folders: ['datasets']
Total images: 4714

Images by folder:
pigmented benign keratosis: 956
melanoma: 908
basal cell carcinoma: 784
nevus: 746
squamous cell carcinoma: 394
vascular lesion: 284
actinic keratosis: 260
dermatofibroma: 222
seborrheic keratosis: 160


In [10]:
from pathlib import Path

root = Path("/kaggle/input/datasets")

print("Metadata files:")
for p in root.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".csv", ".xlsx", ".json", ".txt"}:
        print(p)

print("\nExample image paths:")
for p in images[:5]:
    print(p)

Metadata files:
/kaggle/input/datasets/amirsalahk/isic-2019-ground-truth/ISIC_2019_Training_GroundTruth.csv
/kaggle/input/datasets/amirsalahk/isic-2019-metadata/ISIC_2019_Training_Metadata.csv

Example image paths:
/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024371.jpg
/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024358.jpg
/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024337.jpg
/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test/pigmented benign keratosis/ISIC_0024382.jpg
/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/Skin cancer ISIC The International Skin I

In [11]:
from collections import Counter, defaultdict

counts = Counter()
names = defaultdict(set)

for p in images:
    split = next(
        (part for part in p.parts if part.lower() in {"train", "test", "validation", "val"}),
        "unknown"
    )
    counts[(split, p.parent.name)] += 1
    names[split].add(p.name.lower())

for split in sorted({s for s, _ in counts}):
    print(f"\n{split}: {sum(n for (s, _), n in counts.items() if s == split)} images")
    for (s, label), n in sorted(counts.items()):
        if s == split:
            print(f"  {label}: {n}")

print("\nSame filenames in Train and Test:",
      len(names["Train"] & names["Test"]))



Test: 236 images
  actinic keratosis: 32
  basal cell carcinoma: 32
  dermatofibroma: 32
  melanoma: 32
  nevus: 32
  pigmented benign keratosis: 32
  seborrheic keratosis: 6
  squamous cell carcinoma: 32
  vascular lesion: 6

Train: 4478 images
  actinic keratosis: 228
  basal cell carcinoma: 752
  dermatofibroma: 190
  melanoma: 876
  nevus: 714
  pigmented benign keratosis: 924
  seborrheic keratosis: 154
  squamous cell carcinoma: 362
  vascular lesion: 278

Same filenames in Train and Test: 18


In [12]:
from collections import defaultdict
from hashlib import sha256

by_split_name = defaultdict(list)

for p in images:
    split = next(
        (part.lower() for part in p.parts
         if part.lower() in {"train", "test"}),
        None
    )
    if split:
        by_split_name[(split, p.name.lower())].append(p)

shared = sorted(
    {name for split, name in by_split_name if split == "train"}
    & {name for split, name in by_split_name if split == "test"}
)

def file_hash(path):
    h = sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for name in shared:
    train_files = by_split_name[("train", name)]
    test_files = by_split_name[("test", name)]
    for train_file in train_files:
        for test_file in test_files:
            same = file_hash(train_file) == file_hash(test_file)
            print(
                name,
                "| Train:", train_file.parent.name,
                "| Test:", test_file.parent.name,
                "| identical bytes:", same
            )

isic_0010512.jpg | Train: nevus | Test: actinic keratosis | identical bytes: True
isic_0010512.jpg | Train: nevus | Test: actinic keratosis | identical bytes: True
isic_0010512.jpg | Train: nevus | Test: actinic keratosis | identical bytes: True
isic_0010512.jpg | Train: nevus | Test: actinic keratosis | identical bytes: True
isic_0010803.jpg | Train: melanoma | Test: seborrheic keratosis | identical bytes: True
isic_0010803.jpg | Train: melanoma | Test: seborrheic keratosis | identical bytes: True
isic_0010803.jpg | Train: melanoma | Test: seborrheic keratosis | identical bytes: True
isic_0010803.jpg | Train: melanoma | Test: seborrheic keratosis | identical bytes: True
isic_0010808.jpg | Train: melanoma | Test: seborrheic keratosis | identical bytes: True
isic_0010808.jpg | Train: melanoma | Test: seborrheic keratosis | identical bytes: True
isic_0010808.jpg | Train: melanoma | Test: seborrheic keratosis | identical bytes: True
isic_0010808.jpg | Train: melanoma | Test: seborrheic ke

In [13]:
from collections import defaultdict, Counter

by_id = defaultdict(list)
for p in images:
    by_id[p.name.lower()].append(p)

repeated = {k: v for k, v in by_id.items() if len(v) > 1}
conflicting = {
    k: v for k, v in repeated.items()
    if len({p.parent.name for p in v}) > 1
}

print("Total files:", len(images))
print("Unique filenames:", len(by_id))
print("Repeated filenames:", len(repeated))
print("IDs with conflicting class labels:", len(conflicting))

print("\nConflicting IDs (first 30):")
for name, paths in list(sorted(conflicting.items()))[:30]:
    locations = Counter(
        (next((part for part in p.parts if part in {"Train", "Test"}), "?"),
         p.parent.name)
        for p in paths
    )
    print(name, dict(locations))

Total files: 4714
Unique filenames: 2202
Repeated filenames: 2202
IDs with conflicting class labels: 155

Conflicting IDs (first 30):
isic_0010512.jpg {('Test', 'actinic keratosis'): 2, ('Train', 'nevus'): 2}
isic_0010803.jpg {('Test', 'seborrheic keratosis'): 2, ('Train', 'melanoma'): 2}
isic_0010808.jpg {('Test', 'seborrheic keratosis'): 2, ('Train', 'melanoma'): 2}
isic_0010809.jpg {('Test', 'seborrheic keratosis'): 2, ('Train', 'melanoma'): 2}
isic_0010810.jpg {('Train', 'melanoma'): 2, ('Train', 'seborrheic keratosis'): 2}
isic_0010812.jpg {('Train', 'melanoma'): 2, ('Train', 'seborrheic keratosis'): 2}
isic_0010834.jpg {('Train', 'melanoma'): 2, ('Train', 'seborrheic keratosis'): 2}
isic_0010852.jpg {('Train', 'melanoma'): 2, ('Train', 'seborrheic keratosis'): 2}
isic_0010860.jpg {('Train', 'melanoma'): 2, ('Train', 'seborrheic keratosis'): 2}
isic_0010862.jpg {('Train', 'melanoma'): 2, ('Train', 'seborrheic keratosis'): 2}
isic_0010863.jpg {('Train', 'melanoma'): 2, ('Train', 's

### Preliminary dataset audit

The Kaggle dataset contains 4,714 image files but only 2,202 unique filenames. We found 155 image IDs assigned to more than one diagnostic class. Eighteen image IDs also occur in both the supplied Train and Test folders; the checked copies have identical file contents. Therefore, the supplied labels and Train/Test split will not be used for model evaluation until diagnoses are checked against an authoritative source and a clean split is created.

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

url = (
    "https://isic-archive.s3.amazonaws.com/challenges/2019/"
    "ISIC_2019_Training_GroundTruth.csv"
)

gt = pd.read_csv(url)
print("Official columns:", gt.columns.tolist())

label_cols = [c for c in gt.columns if c != "image"]
official = {}

for _, row in gt.iterrows():
    positive = [c for c in label_cols if row[c] == 1]
    if len(positive) == 1:
        official[str(row["image"]).upper()] = positive[0]

by_id = defaultdict(list)
for p in images:
    by_id[p.stem.upper()].append(p)

matched = {image_id: paths for image_id, paths in by_id.items()
           if image_id in official}
unmatched = set(by_id) - set(matched)
conflicting = {
    image_id: paths for image_id, paths in matched.items()
    if len({p.parent.name for p in paths}) > 1
}

print("\nUnique Kaggle image IDs:", len(by_id))
print("IDs matched to official ISIC 2019 labels:", len(matched))
print("IDs not found in ISIC 2019:", len(unmatched))
print("Matched IDs with conflicting Kaggle labels:", len(conflicting))

print("\nKaggle folder label → official ISIC 2019 label:")
comparison = Counter(
    (p.parent.name, official[image_id])
    for image_id, paths in matched.items()
    for p in paths[:1]
)
for pair, n in sorted(comparison.items()):
    print(f"{pair[0]} → {pair[1]}: {n}")

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*.csv"):
    print(p)

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

csv_path = Path(
    "/kaggle/input/datasets/amirsalahk/isic-2019-ground-truth/"
    "ISIC_2019_Training_GroundTruth.csv"
)

gt = pd.read_csv(csv_path)
label_cols = [c for c in gt.columns if c != "image"]

official = {}
for _, row in gt.iterrows():
    labels = [c for c in label_cols if row[c] == 1]
    if len(labels) == 1:
        official[str(row["image"]).upper()] = labels[0]

by_id = defaultdict(list)
for p in images:
    by_id[p.stem.upper()].append(p)

matched = set(by_id) & set(official)
unmatched = set(by_id) - set(official)

print("Official label columns:", label_cols)
print("Unique Kaggle image IDs:", len(by_id))
print("Matched to ISIC 2019:", len(matched))
print("Not found in ISIC 2019:", len(unmatched))

print("\nKaggle folder → official label (unique image IDs):")
comparison = Counter(
    (kaggle_label, official[image_id])
    for image_id in matched
    for kaggle_label in {p.parent.name for p in by_id[image_id]}
)
for (kaggle_label, official_label), count in sorted(comparison.items()):
    print(f"{kaggle_label} → {official_label}: {count}")

print("\nExamples with conflicting Kaggle folder labels:")
shown = 0
for image_id in sorted(matched):
    kaggle_labels = {p.parent.name for p in by_id[image_id]}
    if len(kaggle_labels) > 1:
        print(image_id, sorted(kaggle_labels), "→ official:", official[image_id])
        shown += 1
        if shown == 10:
            break


In [ ]:
from collections import Counter

print("Verified images by official ISIC 2019 class:")
for label, count in sorted(Counter(official[i] for i in matched).items()):
    print(f"{label}: {count}")

print("\nImages not found in ISIC 2019, by Kaggle folder:")
for label, count in sorted(Counter(
    p.parent.name
    for image_id in unmatched
    for p in by_id[image_id][:1]
).items()):
    print(f"{label}: {count}")

In [ ]:
from pathlib import Path
import pandas as pd

metadata_files = list(
    Path("/kaggle/input").rglob("ISIC_2019_Training_Metadata.csv")
)
print("Files found:", metadata_files)

if metadata_files:
    metadata = pd.read_csv(metadata_files[0])
    print("Columns:", metadata.columns.tolist())
    print(metadata.head(3).to_string(index=False))

In [ ]:
m = metadata.copy()
m["image"] = m["image"].astype(str).str.upper()
m = m.drop_duplicates(subset="image").set_index("image")

verified_ids = sorted(matched)
lesion_ids = m.reindex(verified_ids)["lesion_id"]

known = lesion_ids.notna()
group_sizes = lesion_ids[known].value_counts()

print("Verified image IDs:", len(verified_ids))
print("With lesion_id:", int(known.sum()))
print("Without lesion_id:", int((~known).sum()))
print("Distinct known lesions:", len(group_sizes))
print("Lesions represented by multiple images:", int((group_sizes > 1).sum()))
print("Images in those multi-image lesions:",
      int(group_sizes[group_sizes > 1].sum()))

In [ ]:
from hashlib import sha256

def digest(path):
    h = sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

different_copies = []

for image_id in sorted(matched):
    paths = by_id[image_id]
    if len(paths) > 1 and len({digest(p) for p in paths}) > 1:
        different_copies.append(image_id)

print("Officially labelled IDs checked:", len(matched))
print("IDs whose copies have different bytes:", len(different_copies))
print("First examples:", different_copies[:10])

In [ ]:
from collections import defaultdict
from hashlib import sha256
import pandas as pd

def image_hash(path):
    h = sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

rows = []
for image_id in sorted(matched):
    path = sorted(by_id[image_id], key=str)[0]  # one identical copy
    lesion = m.at[image_id, "lesion_id"] if image_id in m.index else None
    rows.append({
        "image_id": image_id,
        "path": str(path),
        "label": official[image_id],
        "lesion_id": lesion,
        "hash": image_hash(path),
    })

audit = pd.DataFrame(rows)

same_content = audit[audit.duplicated("hash", keep=False)]
mixed_lesions = (
    audit.dropna(subset=["lesion_id"])
         .groupby("lesion_id")["label"]
         .nunique()
)
mixed_lesions = mixed_lesions[mixed_lesions > 1]

print("Unique image IDs:", len(audit))
print("Different IDs sharing identical image content:",
      same_content["image_id"].nunique())
print("Lesion IDs with multiple official diagnoses:", len(mixed_lesions))
print("\nSame-content examples:")
print(same_content[["image_id", "label", "lesion_id"]].head(10).to_string(index=False))
print("\nMixed-lesion examples:")
print(mixed_lesions.head(10).to_string())

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

df = audit.copy()
df["path"] = df["path"].astype(str)

# الصور التي لا تملك lesion_id تحصل على مجموعة مستقلة لكل image_id
df["group"] = df.apply(
    lambda row: (
        f"lesion:{str(row['lesion_id']).strip()}"
        if pd.notna(row["lesion_id"]) and str(row["lesion_id"]).strip()
        else f"image:{row['image_id']}"
    ),
    axis=1,
)

# 20% للاختبار
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
trainval_idx, test_idx = next(outer.split(df, df["label"], df["group"]))
trainval = df.iloc[trainval_idx].copy()
test = df.iloc[test_idx].copy()

# 25% من الباقي للتحقق: تقريبًا 20% من المجموعة الكاملة
inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
train_idx, val_idx = next(
    inner.split(trainval, trainval["label"], trainval["group"])
)
train = trainval.iloc[train_idx].copy()
val = trainval.iloc[val_idx].copy()

train["split"] = "train"
val["split"] = "val"
test["split"] = "test"
splits = pd.concat([train, val, test], ignore_index=True)

assert splits["image_id"].is_unique
assert not (set(train["group"]) & set(val["group"]))
assert not (set(train["group"]) & set(test["group"]))
assert not (set(val["group"]) & set(test["group"]))

print(pd.crosstab(splits["label"], splits["split"], margins=True))
print("\nPercentages:")
print((splits["split"].value_counts(normalize=True) * 100).round(1))

output = Path("/kaggle/working/isic2019_verified_split.csv")
splits.to_csv(output, index=False)
print("\nSaved:", output)

In [ ]:
from PIL import Image
import tensorflow as tf

bad_images = []

for _, row in splits.iterrows():
    try:
        with Image.open(row["path"]) as image:
            image.verify()
    except Exception as error:
        bad_images.append((row["image_id"], str(error)))

print("Unreadable images:", len(bad_images))
print("Examples:", bad_images[:5])
print("GPUs:", tf.config.list_physical_devices("GPU"))


In [16]:
from pathlib import Path
import pandas as pd

file = Path("/kaggle/working/isic2019_verified_split.csv")
print("Split file exists:", file.exists())

if file.exists():
    splits = pd.read_csv(file)
    print("Loaded rows:", len(splits))
    print(splits["split"].value_counts())

Split file exists: False


In [17]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

root = Path("/kaggle/input")
gt_file = next(root.rglob("ISIC_2019_Training_GroundTruth.csv"))
meta_file = next(root.rglob("ISIC_2019_Training_Metadata.csv"))

gt = pd.read_csv(gt_file)
meta = pd.read_csv(meta_file)
classes = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]

gt["image"] = gt["image"].str.upper()
meta["image"] = meta["image"].str.upper()
gt = gt.set_index("image")
meta = meta.drop_duplicates("image").set_index("image")

# نختار نسخة واحدة فقط لكل image ID
paths = {}
for p in sorted(root.rglob("*")):
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        image_id = p.stem.upper()
        if image_id in gt.index and image_id not in paths:
            paths[image_id] = str(p)

rows = []
for image_id, path in sorted(paths.items()):
    positives = [c for c in classes if gt.at[image_id, c] == 1]
    if len(positives) != 1:
        continue
    lesion = meta.at[image_id, "lesion_id"] if image_id in meta.index else None
    group = (
        f"lesion:{lesion}"
        if pd.notna(lesion) and str(lesion).strip()
        else f"image:{image_id}"
    )
    rows.append((image_id, path, positives[0], lesion, group))

df = pd.DataFrame(rows, columns=["image_id", "path", "label", "lesion_id", "group"])
assert len(df) == 1761, f"Expected 1761 images, found {len(df)}"

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
trainval_idx, test_idx = next(outer.split(df, df["label"], df["group"]))
trainval, test = df.iloc[trainval_idx].copy(), df.iloc[test_idx].copy()

inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
train_idx, val_idx = next(inner.split(
    trainval, trainval["label"], trainval["group"]
))
train = trainval.iloc[train_idx].copy()
val = trainval.iloc[val_idx].copy()

train["split"], val["split"], test["split"] = "train", "val", "test"
splits = pd.concat([train, val, test], ignore_index=True)

assert not (set(train.group) & set(val.group))
assert not (set(train.group) & set(test.group))
assert not (set(val.group) & set(test.group))

output = Path("/kaggle/working/isic2019_verified_split.csv")
splits.to_csv(output, index=False)

print(pd.crosstab(splits["label"], splits["split"], margins=True))
print("Saved:", output)


split  test  train  val   All
label                        
AK       25     78   26   129
BCC      84    236   72   392
BKL      89    295   94   478
DF       24     58   22   104
MEL      30     96   28   154
NV       37     99   34   170
SCC      32    110   50   192
VASC     35     78   29   142
All     356   1050  355  1761
Saved: /kaggle/working/isic2019_verified_split.csv


In [19]:
from IPython.display import FileLink, display
display(FileLink("/kaggle/working/isic2019_verified_split.csv"))

/kaggle/working/isic2019_verified_split.csv

In [20]:
import tensorflow as tf

class_names = sorted(splits["label"].unique())
class_to_num = {name: i for i, name in enumerate(class_names)}
print("Classes:", class_to_num)

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [128, 128])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(split_name, shuffle=False):
    part = splits[splits["split"] == split_name]
    labels = part["label"].map(class_to_num).astype("int32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices(
        (part["path"].astype(str).to_numpy(), labels)
    )
    if shuffle:
        ds = ds.shuffle(len(part), seed=42, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset("train", shuffle=True)
val_ds = make_dataset("val")
test_ds = make_dataset("test")

images_batch, labels_batch = next(iter(train_ds))
print("Batch shape:", images_batch.shape)
print("Labels shape:", labels_batch.shape)

Classes: {'AK': 0, 'BCC': 1, 'BKL': 2, 'DF': 3, 'MEL': 4, 'NV': 5, 'SCC': 6, 'VASC': 7}


I0000 00:00:1790272301.008195      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790272301.011232      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Batch shape: (32, 128, 128, 3)
Labels shape: (32,)


In [21]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(42)

train_labels = splits.loc[splits["split"] == "train", "label"].map(class_to_num)
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=train_labels.to_numpy(),
)
class_weights = {i: float(w) for i, w in enumerate(weights)}
print("Class weights:", class_weights)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(128, 128, 3)),

    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/baseline_no_aug.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks,
)

print("Best validation loss:", min(history.history["val_loss"]))
print("Epochs trained:", len(history.history["loss"]))


Class weights: {0: 1.6826923076923077, 1: 0.5561440677966102, 2: 0.4449152542372881, 3: 2.2629310344827585, 4: 1.3671875, 5: 1.3257575757575757, 6: 1.1931818181818181, 7: 1.6826923076923077}
Epoch 1/25
 7/33 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.1135 - loss: 2.2352

I0000 00:00:1790272386.335356     185 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


33/33 ━━━━━━━━━━━━━━━━━━━━ 16s 293ms/step - accuracy: 0.0829 - loss: 2.0820 - val_accuracy: 0.2761 - val_loss: 2.0793
Epoch 2/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.1238 - loss: 2.0777 - val_accuracy: 0.0789 - val_loss: 2.0877
Epoch 3/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 5s 102ms/step - accuracy: 0.1390 - loss: 1.9911 - val_accuracy: 0.1972 - val_loss: 1.9110
Epoch 4/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step - accuracy: 0.2105 - loss: 1.8112 - val_accuracy: 0.1465 - val_loss: 1.8348
Epoch 5/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - accuracy: 0.1895 - loss: 1.7222 - val_accuracy: 0.1634 - val_loss: 2.0763
Epoch 6/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - accuracy: 0.1771 - loss: 1.8267 - val_accuracy: 0.1887 - val_loss: 1.7775
Epoch 7/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - accuracy: 0.1933 - loss: 1.7295 - val_accuracy: 0.1944 - val_loss: 1.8037
Epoch 8/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - accuracy: 0.2286 - loss: 1.7154 - val_accuracy: 0.2394 - val_loss